In [1]:
import pandas as pd 
import numpy as np

In [2]:
df= pd.read_csv('date_fruit_dataset.csv')

df.head() 


,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [3]:
X= df.drop("Class",axis=1)
y= df["Class"]

In [4]:
y.unique() # numbr of unique classes

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [5]:
from sklearn.preprocessing  import StandardScaler,LabelEncoder
leb= LabelEncoder()
y= leb.fit_transform(y)

In [6]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42)

In [7]:
std= StandardScaler()
x_train_scaled= std.fit_transform(X_train)
x_test_scaled=std.transform(X_test)

#### Deepl Learning part

In [8]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


##### create tensors

In [9]:
X_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)

x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)


#### create datasets 

In [10]:
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)


#### DataLoader

In [11]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader= DataLoader(test_dataset,batch_size=32)

### Build our model

In [16]:
import torch.nn as nn

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # 1st Hidden Layer
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),

            # 2nd Hidden Layer
            nn.Linear(64, 64),
            nn.ReLU(),

            # Output Layer - 7 classes
            nn.Linear(64, 7)
        )

    def forward(self, x):
        return self.model(x)

In [18]:
model = ANN()

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [20]:
# TRAINING THE NN

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)

        loss = criterion(outputs, yb)

        loss.backward()
        optimizer.step()  # parameter update step

        running_loss += loss.item()

    training_loss = running_loss / len(train_loader)

    print(
        f"Epoch -> {epoch + 1}/{epochs}, "
        f"Loss -> {training_loss:.4f}"
    )

Epoch -> 1/100, Loss -> 1.7280
Epoch -> 2/100, Loss -> 1.1325
Epoch -> 3/100, Loss -> 0.7592
Epoch -> 4/100, Loss -> 0.5712
Epoch -> 5/100, Loss -> 0.4489
Epoch -> 6/100, Loss -> 0.3857
Epoch -> 7/100, Loss -> 0.3391
Epoch -> 8/100, Loss -> 0.3113
Epoch -> 9/100, Loss -> 0.2827
Epoch -> 10/100, Loss -> 0.2510
Epoch -> 11/100, Loss -> 0.2454
Epoch -> 12/100, Loss -> 0.2206
Epoch -> 13/100, Loss -> 0.2190
Epoch -> 14/100, Loss -> 0.2150
Epoch -> 15/100, Loss -> 0.2082
Epoch -> 16/100, Loss -> 0.1906
Epoch -> 17/100, Loss -> 0.1780
Epoch -> 18/100, Loss -> 0.1716
Epoch -> 19/100, Loss -> 0.1610
Epoch -> 20/100, Loss -> 0.1548
Epoch -> 21/100, Loss -> 0.1517
Epoch -> 22/100, Loss -> 0.1477
Epoch -> 23/100, Loss -> 0.1417
Epoch -> 24/100, Loss -> 0.1397
Epoch -> 25/100, Loss -> 0.1332
Epoch -> 26/100, Loss -> 0.1379
Epoch -> 27/100, Loss -> 0.1263
Epoch -> 28/100, Loss -> 0.1249
Epoch -> 29/100, Loss -> 0.1202
Epoch -> 30/100, Loss -> 0.1149
Epoch -> 31/100, Loss -> 0.1158
Epoch -> 32/100, 

In [21]:
# Evaluation
model.eval()

total=0
correct=0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs=model(xb) 
        # print(outputs)
        _,index_pred=torch.max(outputs,1) # willl retunr the max value and their index
        
        correct+=(index_pred==yb).sum().item() # for all batches predicted value  
        total+=yb.size(0) # actual sample in each batch

print(f"Total Values : {total}")
print(f"Predicted Values :{correct}")
print(f" Accuracy : {(correct/total)* 100  }")
        

Total Values : 180
Predicted Values :173
 Accuracy : 96.11111111111111


In [ ]:
# Apply PCA TO REDUCE THE DIMENTIONALITY REDUCTIO TO INCREASE THE ACCURACY
